# PM-3 — Weekly Clustering & Risk Scoring Pipeline

**Goal:** Identify outlets with rising temperature-difference trends using KMeans clustering on weekly session data, then rank them by proximity to a "failure-like" cluster.

### Pipeline Overview
| Step | Description |
|------|-------------|
| 1 | Imports & configuration |
| 2 | Load session data |
| 3 | Parse IDs & dates |
| 4 | Weekly aggregation + filtering |
| 5 | Outlier capping & timeline alignment |
| 6 | Z-score normalization → feature matrix |
| 7 | Enriched outlet-level features |
| 8 | KMeans model selection |
| 9 | Cluster assignment & failure-like identification |
| 10 | Risk scoring (distance + slopes) |
| 11 | Cluster visualizations |
| 12 | Daily panel plots for top risky outlets |

## Step 1 — Imports & Configuration

In [ ]:
# Core
import os
from pathlib import Path

# Data
import numpy as np
import pandas as pd

# Modeling
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Plots
import matplotlib.pyplot as plt

print("✅ Imports loaded")

# === PATH CONFIGURATION ===
BASE_DIR = Path.cwd().parent

DATA_DIR = BASE_DIR / "data_all"
OUT_DIR  = BASE_DIR / "outputs_pm3"

# Input files — only Sessions needed for this pipeline
SESS_FILE = DATA_DIR / "SeccSessionStop_last_year.csv"

# Parameters
MIN_WEEKS    = 26          # minimum weeks to keep an outlet in clustering
OUTLIER_Q    = 0.99        # weekly 99th-percentile cap for temp diff
K_RANGE      = list(range(3, 9))   # KMeans k values to evaluate
TOP_N        = 20          # how many risky outlets to export

# Create output directories
for sub in ["cluster_plots", "top20_plots", "daily_plots", "cluster_exports"]:
    (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)

print("✅ Config loaded")
print(f"   DATA_DIR: {DATA_DIR}  (exists: {DATA_DIR.exists()})")
print(f"   OUT_DIR:  {OUT_DIR}")

## Step 2 — Load Session Data

Only the **SeccSessionStop** file is needed for this clustering pipeline.  
CTD and PLE are used in the rule-based notebook (`02_exploratory_analysis_0210_all_outlets_NewScoring`).

In [ ]:
df_sess = pd.read_csv(SESS_FILE, low_memory=False)
print(f"✅ Loaded sessions: shape={df_sess.shape}")
print(f"   Columns: {list(df_sess.columns)}")
display(df_sess.head(3))

## Step 3 — Parse IDs & Dates

- Rename columns to standard names (`@logStream`, `date`, `Sess_temp_diff_mean`)
- Parse timestamps, drop invalid rows
- Split `IDOutlet` into `charger_id` (all but last char) and `outlet` (last digit)
- Convert `duration` from **ms → minutes** before any aggregation

In [ ]:
# Rename to standard column names
df_sess = df_sess.rename(columns={
    "IDOutlet": "@logStream",
    "@timestamp": "date",
    "diff": "Sess_temp_diff_mean",
})

# Keep only needed columns
keep_cols = ["@logStream", "date", "Sess_temp_diff_mean"]
for col in ["energy", "duration"]:
    if col in df_sess.columns:
        keep_cols.append(col)
df_sess = df_sess[keep_cols].copy()

# Parse dates
df_sess["date"] = pd.to_datetime(df_sess["date"], errors="coerce")
n_bad = df_sess["date"].isna().sum()
df_sess = df_sess.dropna(subset=["date"])
print(f"✅ Parsed dates — dropped {n_bad} invalid timestamps")

# Convert duration ms → minutes (before any aggregation)
if "duration" in df_sess.columns:
    df_sess["duration"] = pd.to_numeric(df_sess["duration"], errors="coerce")
    df_sess["duration_min"] = df_sess["duration"] / 60_000.0

# Split charger_id / outlet from @logStream (last char = outlet digit)
df_sess["charger_id"] = df_sess["@logStream"].astype(str).str[:-1]
df_sess["outlet"]     = df_sess["@logStream"].astype(str).str[-1]

print(f"✅ Rows: {len(df_sess):,}  |  Unique outlets: {df_sess['@logStream'].nunique():,}")
display(df_sess.head())

## Step 4 — Weekly Aggregation & Filtering

1. Clean non-numeric `Sess_temp_diff_mean` values
2. Aggregate to **weekly mean** per outlet
3. Drop outlets with fewer than `MIN_WEEKS` (26) weeks of data

In [ ]:
# 1) Clean numeric
df_clean = df_sess.dropna(subset=["Sess_temp_diff_mean"]).copy()
df_clean["Sess_temp_diff_mean"] = pd.to_numeric(df_clean["Sess_temp_diff_mean"], errors="coerce")
df_clean = df_clean.dropna(subset=["Sess_temp_diff_mean"])

# 2) Weekly aggregation (mean per outlet per week)
df_clean["week"] = df_clean["date"].dt.to_period("W").dt.start_time

df_weekly = (
    df_clean
    .groupby(["charger_id", "outlet", "week"], as_index=False)["Sess_temp_diff_mean"]
    .mean()
    .sort_values(["charger_id", "outlet", "week"])
)
print(f"✅ Weekly aggregation: {df_weekly.shape}")

# 3) Filter outlets with insufficient weeks
valid_counts = (
    df_weekly.groupby(["charger_id", "outlet"])["week"]
    .count()
    .reset_index(name="n_weeks")
)
valid_pairs = valid_counts[valid_counts["n_weeks"] >= MIN_WEEKS][["charger_id", "outlet"]]
df_weekly = df_weekly.merge(valid_pairs, on=["charger_id", "outlet"], how="inner")

n_outlets = df_weekly[["charger_id", "outlet"]].drop_duplicates().shape[0]
print(f"✅ Outlets after filtering (≥{MIN_WEEKS} weeks): {n_outlets:,}")

## Step 5 — Outlier Capping & Timeline Alignment

- Cap each outlet's weekly values at the **99th percentile** to reduce noise
- Align all outlets to the **same weekly timeline** (interpolate missing weeks)

In [ ]:
# 4) Outlier capping per outlet (99th percentile)
def cap_out(group):
    cap = group["Sess_temp_diff_mean"].quantile(OUTLIER_Q)
    group["Sess_temp_diff_mean"] = np.clip(group["Sess_temp_diff_mean"], None, cap)
    return group

df_weekly = (
    df_weekly
    .groupby(["charger_id", "outlet"], group_keys=False)
    .apply(cap_out, include_groups=False)
    .reset_index()
)
print("✅ Outlier capping applied.")

# 5) Align all outlets to the same weekly timeline
min_week = df_weekly["week"].min()
max_week = df_weekly["week"].max()
full_weeks = pd.date_range(start=min_week, end=max_week, freq="W-MON")

aligned = []
for (cid, out), g in df_weekly.groupby(["charger_id", "outlet"]):
    g = g.set_index("week").reindex(full_weeks)
    g["Sess_temp_diff_mean"] = g["Sess_temp_diff_mean"].interpolate(limit_direction="both")
    g["charger_id"] = cid
    g["outlet"] = out
    aligned.append(g.reset_index().rename(columns={"index": "week"}))

df_aligned = pd.concat(aligned, ignore_index=True)
print(f"✅ Week alignment completed: {df_aligned.shape}")

## Step 6 — Z-Score Normalization → Feature Matrix

- Z-score each outlet's time series so clustering compares **shape**, not absolute level
- Pivot into a matrix: rows = outlets, columns = weeks

In [ ]:
# 6) Z-score normalization per outlet
def zscore_outlet(group):
    x = group["Sess_temp_diff_mean"].astype(float).values
    mu, sd = np.nanmean(x), np.nanstd(x)
    group["z"] = (x - mu) / sd if (sd > 0 and not np.isnan(sd)) else 0.0
    return group

df_aligned = (
    df_aligned
    .groupby(["charger_id", "outlet"], group_keys=False)
    .apply(zscore_outlet, include_groups=False)
    .reset_index()
)
print("✅ Z-normalization complete.")

# 7) Build weekly feature matrix (z-series)
mat = (
    df_aligned
    .pivot_table(index=["charger_id", "outlet"], columns="week", values="z")
    .sort_index(axis=1)
    .fillna(0.0)
)

print(f"✅ Weekly feature matrix: {mat.shape}  (outlets × weeks)")
display(mat.head())

## Step 7 — Enriched Outlet-Level Features

Add **absolute** summary features alongside the z-score time series to give the clustering more signal:

| Feature | Description |
|---------|-------------|
| `mean_temp` | Overall mean weekly temp diff |
| `max_temp` | Max weekly temp diff |
| `weeks_with_data` | Total weeks of data |
| `high_weeks_ratio` | Fraction of weeks with temp diff > 5°C |
| `slope_8w` | Linear slope over last 8 weeks |

In [ ]:
def slope_last_n(values, n=8):
    """Linear slope over the last n values."""
    values = np.array(values, dtype=float)
    if len(values) < n:
        return 0.0
    sub = values[-n:]
    x = np.arange(len(sub))
    denom = np.sum((x - x.mean()) ** 2)
    return np.sum((x - x.mean()) * (sub - sub.mean())) / denom if denom else 0.0


base_df = df_aligned.copy()

# Basic summary stats
summary = (
    base_df.groupby(["charger_id", "outlet"])
    .agg(
        mean_temp=("Sess_temp_diff_mean", "mean"),
        max_temp=("Sess_temp_diff_mean", "max"),
        weeks_with_data=("Sess_temp_diff_mean", "count"),
    )
    .reset_index()
)

# High-temperature ratio (>5°C)
THR = 5.0
high_ratio = (
    base_df.assign(high=(base_df["Sess_temp_diff_mean"] > THR).astype(int))
    .groupby(["charger_id", "outlet"])["high"]
    .mean()
    .reset_index(name="high_weeks_ratio")
)
summary = summary.merge(high_ratio, on=["charger_id", "outlet"], how="left")

# Slope over last 8 weeks
slopes = []
for (cid, outlet), sub in base_df.sort_values("week").groupby(["charger_id", "outlet"]):
    slopes.append({
        "charger_id": cid, "outlet": outlet,
        "slope_8w": slope_last_n(sub["Sess_temp_diff_mean"].values)
    })
summary = summary.merge(pd.DataFrame(slopes), on=["charger_id", "outlet"], how="left")

print("✅ Enriched outlet features:")
display(summary.head())

# Combine: z-score matrix + enriched features → final feature matrix
summary_aligned = (
    summary.set_index(["charger_id", "outlet"])
    .reindex(mat.index)
    .fillna(0.0)
)

X_base = mat.to_numpy(float)
X_extra = summary_aligned.to_numpy(float)
X_enriched = np.hstack([X_base, X_extra])

print(f"✅ Enriched feature matrix: {X_enriched.shape}  (outlets × [weeks + summary features])")

## Step 8 — KMeans Model Selection

Run KMeans for $k \in [3, 8]$ and pick the best $k$ by **silhouette score**.  
Also track the **Davies-Bouldin index** (lower = better).

In [ ]:
X = X_enriched
results = []
best_sil, best_model, best_k = -1, None, None

for k in K_RANGE:
    km = KMeans(n_clusters=k, n_init="auto", random_state=42)
    labels = km.fit_predict(X)

    sil = silhouette_score(X, labels)
    dbi = davies_bouldin_score(X, labels)
    results.append({"k": k, "silhouette": sil, "davies_bouldin": dbi})
    print(f"  k={k}  →  silhouette={sil:.4f}  |  DB={dbi:.4f}")

    if sil > best_sil:
        best_sil, best_model, best_k = sil, km, k

print(f"\n✅ Best K = {best_k}  (silhouette = {best_sil:.4f})")

# Save metrics
df_modelsel = pd.DataFrame(results)
df_modelsel.to_csv(OUT_DIR / "model_selection_metrics.csv", index=False)

## Step 9 — Cluster Assignment & Failure-Like Identification

- Assign final cluster labels using the best model
- Identify the **failure-like cluster** = the one whose centroid has the **steepest upward slope** over time (rising temp diff = degradation signal)

In [ ]:
labels = best_model.labels_
centers = best_model.cluster_centers_

# Assign cluster labels
df_clusters = mat.copy()
df_clusters["cluster"] = labels
df_clusters = df_clusters.reset_index()
print("✅ Cluster labels assigned.")

# Identify failure-like cluster (highest positive slope in weekly part)
def slope_of_vector(v: np.ndarray) -> float:
    x = np.arange(len(v))
    denom = np.sum((x - x.mean()) ** 2)
    return np.sum((x - x.mean()) * (v - v.mean())) / denom if denom else 0.0

n_weeks = mat.shape[1]
cluster_slopes = []
for c in range(best_k):
    slope = slope_of_vector(centers[c, :n_weeks])
    cluster_slopes.append({"cluster": c, "slope": round(slope, 6)})

df_cshape = pd.DataFrame(cluster_slopes).sort_values("slope", ascending=False)
print("✅ Cluster slope summary:")
display(df_cshape)

failure_like = int(df_cshape.iloc[0]["cluster"])
print(f"⚡ Failure-like cluster: {failure_like}  (highest rising trend)")

## Step 10 — Risk Scoring

Composite risk score (0–100) for **every** outlet:

$$\text{risk} = 0.6 \times (1 - d_{\text{norm}}) + 0.2 \times s_{24\text{w}} + 0.2 \times s_{12\text{w}}$$

where:
- $d_{\text{norm}}$ = min-max normalized distance to the failure-like centroid (closer → riskier)
- $s_{12\text{w}}, s_{24\text{w}}$ = min-max normalized linear slopes over the last 12 / 24 features

In [ ]:
from numpy.linalg import norm

def outlet_slope(vec, last_n):
    if len(vec) < last_n:
        return 0.0
    sub = vec[-last_n:]
    x = np.arange(len(sub))
    denom = np.sum((x - x.mean()) ** 2)
    return np.sum((x - x.mean()) * (sub - sub.mean())) / denom if denom else 0.0

def minmax(x):
    a, b = np.nanmin(x), np.nanmax(x)
    return (x - a) / (b - a) if a != b else np.zeros_like(x)

# Distance to failure centroid
failure_centroid = centers[failure_like]
dists = norm(X - failure_centroid[None, :], axis=1)

# Slopes over last 12 and 24 features
slopes_12 = np.array([outlet_slope(X[i], 12) for i in range(X.shape[0])])
slopes_24 = np.array([outlet_slope(X[i], 24) for i in range(X.shape[0])])

# Composite risk score
dist_norm = 1 - minmax(dists)   # closer = higher risk
risk = 0.6 * dist_norm + 0.2 * minmax(slopes_24) + 0.2 * minmax(slopes_12)
risk_0_100 = (100 * risk).round(1)

# Store results
df_clusters["distance_to_failure_centroid"] = dists
df_clusters["slope12"] = slopes_12
df_clusters["slope24"] = slopes_24
df_clusters["risk_0_100"] = risk_0_100

df_clusters.to_csv(OUT_DIR / "clusters_with_risk.csv", index=False)
print("✅ Risk scores computed and saved.")
display(df_clusters[["charger_id", "outlet", "cluster", "risk_0_100"]].head(10))

## Step 11 — Cluster Visualizations

Plot each cluster's **mean ± IQR** z-scored temperature-diff curve over weeks.  
Also extract the **top risky outlets** from the failure-like cluster.

In [ ]:
def plot_cluster_mean_iqr(centers, labels, X_week, save_dir: Path):
    """Plot mean ± IQR z-scored time series for each cluster."""
    save_dir.mkdir(exist_ok=True)
    labs = np.array(labels)
    for c in sorted(np.unique(labs)):
        idx = np.where(labs == c)[0]
        if len(idx) == 0:
            continue
        series = X_week[idx, :]
        q25 = np.nanquantile(series, 0.25, axis=0)
        q75 = np.nanquantile(series, 0.75, axis=0)
        mean = np.nanmean(series, axis=0)

        plt.figure(figsize=(7, 4))
        plt.plot(np.arange(mean.size), mean, linewidth=2, label=f"Cluster {c}")
        plt.fill_between(np.arange(mean.size), q25, q75, alpha=0.25, label="IQR")
        plt.axhline(0, color="gray", linewidth=1, linestyle="--")
        plt.title(f"Cluster {c} – Mean ± IQR (n={len(idx)})")
        plt.xlabel("Week position")
        plt.ylabel("Z-scored Temp Diff")
        plt.legend()
        plt.tight_layout()
        path = save_dir / f"cluster_{c}_mean_iqr.png"
        plt.savefig(path, dpi=150)
        plt.close()
        print(f"  ✅ Saved: {path.name}")

X_week = mat.to_numpy(float)
plot_cluster_mean_iqr(centers[:, :n_weeks], labels, X_week, OUT_DIR / "cluster_plots")

# --- Top risky outlets ---
df_fail = df_clusters[df_clusters["cluster"] == failure_like].copy()
df_fail_sorted = df_fail.sort_values("risk_0_100", ascending=False).reset_index(drop=True)

top_outlets = df_fail_sorted.head(TOP_N)[["charger_id", "outlet", "risk_0_100"]]
top_outlets.to_csv(OUT_DIR / "top_risky_outlets.csv", index=False)
print(f"\n✅ Top {TOP_N} risky outlets saved.")
display(top_outlets)

## Step 12 — Daily Panel Plots for Top Risky Outlets

For each top outlet, generate a **3-panel daily plot**:

1. **Max ΔT** (with 5°C threshold line + 30-day rolling mean)
2. **Mean session energy** (Wh)
3. **Mean session duration** (minutes)

In [ ]:
# Build daily aggregates from session data
df_daily = df_sess.copy()
df_daily["day"] = df_daily["date"].dt.floor("D")

# Ensure energy/duration_min columns exist
if "energy" not in df_daily.columns:
    df_daily["energy"] = np.nan
if "duration_min" not in df_daily.columns:
    df_daily["duration_min"] = np.nan

daily_agg = (
    df_daily
    .groupby(["charger_id", "outlet", "day"], as_index=False)
    .agg(
        sess_temp_diff_max=("Sess_temp_diff_mean", "max"),
        sess_temp_diff_mean=("Sess_temp_diff_mean", "mean"),
        energy_mean=("energy", "mean"),
        duration_mean_min=("duration_min", "mean"),
        sess_count=("Sess_temp_diff_mean", "count"),
    )
)

print(f"✅ Daily aggregation: {daily_agg.shape}")
display(daily_agg.head())

In [ ]:
def cap_outliers(series, q=0.98):
    """Cap values at the q-th percentile."""
    s = pd.to_numeric(series, errors="coerce")
    if s.dropna().empty:
        return s
    return np.minimum(s, s.quantile(q))


def plot_daily_panels(daily_df: pd.DataFrame,
                      top_df: pd.DataFrame,
                      out_dir: Path,
                      roll_window: int = 30):
    """Generate 3-panel daily plots for each top outlet."""
    out_dir.mkdir(exist_ok=True)

    for idx, row in top_df.iterrows():
        cid, outlet, risk = row["charger_id"], row["outlet"], row["risk_0_100"]

        sub = daily_df[
            (daily_df["charger_id"] == cid) & (daily_df["outlet"] == outlet)
        ].copy().sort_values("day")

        if sub.empty:
            print(f"⚠️ No daily data for {cid} outlet {outlet}")
            continue

        # Outlier capping
        sub["temp_cap"]   = cap_outliers(sub["sess_temp_diff_max"], q=0.98)
        sub["energy_cap"] = cap_outliers(sub["energy_mean"], q=0.98)
        sub["dur_cap"]    = cap_outliers(sub["duration_mean_min"], q=0.98)
        sub["temp_roll"]  = sub["temp_cap"].rolling(roll_window, min_periods=5).mean()

        fig, axs = plt.subplots(3, 1, figsize=(16, 10), sharex=True)
        fig.suptitle(
            f"Rank {idx+1} | Charger {cid} – Outlet {outlet} | Risk = {risk}",
            fontsize=14,
        )

        # Panel 1: Max ΔT
        axs[0].plot(sub["day"], sub["temp_cap"], marker="o", alpha=0.4,
                    label="Daily max ΔT (°C)")
        axs[0].plot(sub["day"], sub["temp_roll"], "--", color="red", lw=2,
                    label=f"{roll_window}d rolling mean")
        axs[0].axhline(5.0, color="gray", ls="--", lw=1, label="5.0°C threshold")
        axs[0].set_ylabel("ΔT (°C)")
        axs[0].legend(loc="upper left")
        axs[0].grid(alpha=0.2)

        # Panel 2: Energy
        axs[1].plot(sub["day"], sub["energy_cap"], marker="o", alpha=0.4)
        axs[1].set_ylabel("Energy (Wh)")
        axs[1].set_title("Daily mean session energy")
        axs[1].grid(alpha=0.2)

        # Panel 3: Duration
        axs[2].plot(sub["day"], sub["dur_cap"], marker="o", alpha=0.4, color="purple")
        axs[2].set_ylabel("Duration (min)")
        axs[2].set_title("Daily mean session duration")
        axs[2].set_xlabel("Date")
        axs[2].grid(alpha=0.2)

        plt.tight_layout(rect=[0, 0.02, 1, 0.95])
        fname = out_dir / f"Rank_{idx+1:02d}_{cid}_Outlet{outlet}.png"
        plt.savefig(fname, dpi=150)
        plt.close()
        print(f"  ✅ {fname.name}")


# Run
plot_daily_panels(daily_agg, top_outlets, OUT_DIR / "daily_plots")
print("\n🎉 Pipeline complete!")